Build user-item matrix.  Rows = users, columnts = movies. Values = ratings
This matrix is expect to be very sparse
Decompose Matrix into Latent Factors

In [17]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, GridSearchCV
from surprise import accuracy
import pandas as pd
import numpy as np
from collections import defaultdict

In [18]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SVD-Recommender").getOrCreate()  
# create Spark session

In [19]:
from surprise import Dataset, Reader

In [20]:
import pandas as pd
from surprise import Dataset, Reader



In [21]:
ratings = pd.read_csv(
    "/home/rajesh/CSL7100/Assignment3/ml-latest-small/ratings.csv"
)
# load MovieLens ratings dataset

reader = Reader(rating_scale=(0.5,5))
# define rating scale used in dataset

data = Dataset.load_from_df(
    ratings[['userId','movieId','rating']],
    reader
)
# convert pandas dataframe into Surprise dataset format

In [22]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [23]:
reader = Reader(rating_scale=(0.5,5))
# define rating scale used in dataset

data = Dataset.load_from_df(
    ratings[['userId','movieId','rating']],
    reader
)
# convert pandas dataframe into Surprise dataset format

In [24]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data, test_size=0.2)
# split dataset into training and testing sets



In [25]:
from surprise import SVD

model = SVD(
    n_factors=20,   # number of latent factors
    n_epochs=20,    # training iterations
    lr_all=0.005,   # learning rate
    reg_all=0.02    # regularization to prevent overfitting
)
# initialize SVD recommender model

model.fit(trainset)
# train model on training dataset

In [26]:
predictions = model.test(testset)
# generate predictions for all user–movie pairs in test set

In [27]:
from surprise import accuracy
rmse = accuracy.rmse(predictions)
print(f"Test RMSE: {rmse:.4f}")

RMSE: 0.8692
Test RMSE: 0.8692


In [28]:
from collections import defaultdict

# -----------------------------
# 7. Precision@K and Recall@K
# -----------------------------
def precision_recall_at_k(predictions, k=5, threshold=4):
    """Return precision and recall at K metrics for each user"""
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]

        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, true_r) in top_k)
        n_rel_and_rec_k = sum((true_r >= threshold and est >= threshold) for (est, true_r) in top_k)

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    precision_avg = np.mean(list(precisions.values()))
    recall_avg = np.mean(list(recalls.values()))
    return precision_avg, recall_avg






Precision@5: 0.5700
Recall@5: 0.2404


In [31]:
precision, recall = precision_recall_at_k(predictions, k=5, threshold=4)
print(f"Precision@5: {precision:.4f}")
print(f"Recall@5: {recall:.4f}")

precision, recall = precision_recall_at_k(predictions, k=10, threshold=4)
print(f"Precision@10: {precision:.4f}")
print(f"Recall@10: {recall:.4f}")

precision, recall = precision_recall_at_k(predictions, k=20, threshold=4)
print(f"Precision@20: {precision:.4f}")
print(f"Recall@20: {recall:.4f}")

precision, recall = precision_recall_at_k(predictions, k=50, threshold=4)
print(f"Precision@20: {precision:.4f}")
print(f"Recall@20: {recall:.4f}")

Precision@5: 0.5700
Recall@5: 0.2404
Precision@10: 0.5645
Recall@10: 0.2990
Precision@20: 0.5580
Recall@20: 0.3317
Precision@20: 0.5573
Recall@20: 0.3493


Precision@50: 0.5573
Recall@50: 0.3493


In [19]:
# -----------------------------
# 8. Hyperparameter tuning using GridSearchCV
# -----------------------------
param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.05]
}


In [21]:
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3)
gs.fit(data)

print("Best RMSE from GridSearchCV:", gs.best_score['rmse'])
print("Best parameters:", gs.best_params['rmse'])

Best RMSE from GridSearchCV: 0.8706201396492456
Best parameters: {'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05}
